# Implied Volatility

This notebook develops numerical methods for recovering implied volatility from observed option prices.

The objective is to:
1. understand implied volatility as the inverse of the Black-Scholes pricing problem;
2. implement an implied volatility solver;
3. compare numerical root-finding methods;
4. analyse convergence and failure cases;
5. validate recovered volatility against known Black-Scholes prices.

## 1. The Implied Volatility Problem

### From Option Price to Implied Volatility 
 
In the Black-Scholes model, the option price is calculated using the underlying price, strike price, time to maturity, risk-free rate and volatility: 
 
$$
C_{BS} = C(S,K,T,r,\sigma)
$$
 
In practice, the market price of an option is observable, while the volatility implied by that price is not. Implied volatility is therefore the value of $\sigma$ that makes the Black-Scholes price equal to the observed market price: 
 
$$
C_{BS}(\sigma_{IV}) = C_{\text{market}}
$$
 
Equivalently, the problem can be written as finding the root of: 
 
$$
f(\sigma) = C_{BS}(\sigma) - C_{\text{market}} = 0
$$
 
Because volatility cannot be conveniently isolated from the Black-Scholes formula, implied volatility must be estimated numerically.

### Bisection Method 
 
For a European call option, the Black-Scholes price increases with volatility. Therefore, if the market price lies between the option prices generated by two volatility estimates, the implied volatility lies between those estimates. 
 
The bisection method repeatedly: 
 
1. Defines lower and upper volatility bounds.
2. Calculates the midpoint of the interval:

$$
\sigma_{\text{mid}} = \frac{\sigma_{\text{low}} + \sigma_{\text{high}}}{2}
$$

3. Prices the option using the midpoint volatility.
4. Compares the model price with the observed market price.
5. Retains the half of the interval containing the implied volatility.
6. Repeats until the pricing error is sufficiently small.

In [1]:
import sys
from pathlib import Path

import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

from src.black_scholes import black_scholes_price
from src.implied_volatility import Newton_Raphson_iv

In [2]:
market_price = 10.45
sigma_low = 0.10
sigma_high = 0.30

In [3]:
sigma_mid = (sigma_low + sigma_high) / 2

Using the Black Scholes Price function from 01_black_scholes_exploration.ipynb

In [4]:
black_scholes_price(100, 100, 1, 0.05, 0.0, sigma_mid, "call")

np.float64(10.450583572185565)

In [5]:
tolerance = 1e-6

In [6]:
for x in range(100):
    sigma_mid = (sigma_low + sigma_high) / 2

    model_price = black_scholes_price(100, 100, 1, 0.05, 0.0, sigma_mid, "call")

    error = model_price - market_price

    if abs(error) < tolerance:
        break

    if error > 0:
        sigma_high = sigma_mid
    
    else:
        sigma_low = sigma_mid





In [7]:
print(sigma_mid)
print(model_price)
print(market_price)
print(error)

0.1999844551086426
10.450000266333205
10.45
2.6633320615587763e-07


In [8]:
def implied_vol_bisection(S, K, T, r, q, market_price, option_type="call"):

    sigma_high = 2.00
    sigma_low = 0.001
    tolerance = 1e-6
    
    for x in range(100):
        sigma_mid = (sigma_low + sigma_high) / 2
        model_price = black_scholes_price(S,K,T,r,q,sigma_mid,option_type)
        
        error = model_price - market_price
        
        if abs(error) < tolerance:
            break
        
        if error > 0:
            
            sigma_high = sigma_mid
            
        else:
            sigma_low = sigma_mid

    return sigma_mid





In [9]:
implied_vol_bisection(100, 100, 1, 0.05, 0.0, market_price, "call")

0.19998444190621373

In [10]:
true_sigma = 0.30

test_price = black_scholes_price(100,100,1, 0.05,0.0,true_sigma ,"call")

In [11]:
test_price

np.float64(14.231254785985819)

In [12]:
recovered_iv = implied_vol_bisection(100,100,1, 0.05,0.0, test_price ,"call")

In [13]:
recovered_iv

0.299999998062849

## 2. Newton-Raphson Method

The Newton-Raphson method provides a faster approach for solving for implied volatility.

As with bisection, the objective is to find the volatility that makes the Black-Scholes model price equal to the observed market price:

$$
f(\sigma) = C_{BS}(\sigma) - C_{\text{market}} = 0
$$

Rather than repeatedly halving a volatility interval, Newton-Raphson uses the current pricing error and the sensitivity of the option price to volatility to estimate a better volatility guess.

### Vega and the Volatility Update

Vega measures the sensitivity of the option price to a change in volatility:

$$
\text{Vega} = \frac{\partial C}{\partial \sigma}
$$

Therefore, the approximate change in volatility required to correct a pricing error can be written as:

$$
\Delta \sigma \approx \frac{\Delta C}{\text{Vega}}
$$

A large Vega means the option price is highly sensitive to volatility, so a relatively small volatility adjustment may correct the pricing error. A small Vega means a larger volatility adjustment may be required.

### Newton-Raphson Update

Starting from an initial volatility estimate, Newton-Raphson updates the estimate using:

$$
\sigma_{n+1}
=
\sigma_n
-
\frac{C_{BS}(\sigma_n)-C_{\text{market}}}
{\text{Vega}(\sigma_n)}
$$

The numerator is the pricing error:

$$
\text{error} = C_{BS}(\sigma_n)-C_{\text{market}}
$$

The algorithm therefore:

1. Starts with an initial volatility guess.
2. Calculates the Black-Scholes price using that volatility.
3. Calculates the difference between the model price and market price.
4. Calculates Vega at the current volatility.
5. Uses the pricing error and Vega to update the volatility estimate.
6. Repeats until the pricing error is sufficiently small.

In [14]:
Newton_Raphson_iv(100,100,1, 0.05,0.0, market_price ,"call")

np.float64(0.19998444804269158)